# 09 — Advanced: Evaluation

**Stage 9 of the workshop (Production, extended).** Scoring agent outputs against expected behavior — a minimal test-table-style eval harness.

## Problem

Getting from "works on my laptop" to something a team can rely on — you need a repeatable way to know whether an agent still behaves correctly after a prompt, tool, or model-provider change, not just eyeball a few manual runs.

## Concept

Strands doesn't ship an eval framework — evaluation is "run the agent against known cases, score what comes back," same as testing any other piece of software. This is the minimum viable version: a table of `EvalCase`s (name, prompt, a plain Python `check` function), run each, print pass/fail, summarize. Swap the scorer for something LLM-judged, or plug in RAGAS/a hosted eval service, once cases outgrow what a plain Python check can score.

## Architecture

```
CASES: list[EvalCase]
     │
     ▼
for case in CASES:
     output = str(agent(case.prompt))
     ok = case.check(output)          ── contains_any(...) / contains_all(...)
     print [PASS]/[FAIL] case.name
     │
     ▼
print "N/M cases passed"
```

Three cases exercise three different behaviors: correct tool use (`basic_addition`), graceful refusal on unanswerable questions (`refuses_unknown`), and using the tool rather than mental math for a case where a wrong mental calculation is plausible (`uses_tool_not_mental_math`).

## Step 1 — Model, tool, and agent setup

In [ ]:
import sys
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent, tool

model = get_model()


@tool
def add(x: int, y: int) -> int:
    """Add two numbers."""
    return x + y


agent = Agent(model=model, tools=[add], callback_handler=None)


## Step 2 — Define the eval case structure and scoring helpers

In [ ]:
@dataclass
class EvalCase:
    name: str
    prompt: str
    check: Callable[[str], bool]


def contains_any(*substrings: str) -> Callable[[str], bool]:
    return lambda output: any(s.lower() in output.lower() for s in substrings)


def contains_all(*substrings: str) -> Callable[[str], bool]:
    return lambda output: all(s.lower() in output.lower() for s in substrings)


## Step 3 — Define the test cases

In [ ]:
CASES = [
    EvalCase("basic_addition", "What is 12 plus 30? Use the add tool.", contains_all("42")),
    EvalCase(
        "refuses_unknown",
        "What is the capital of Mars?",
        contains_any("don't know", "no capital", "not know", "doesn't have", "does not have"),
    ),
    EvalCase("uses_tool_not_mental_math", "What is 999 plus 1? Use the add tool.", contains_all("1000")),
]


## Step 4 — Run the eval harness

In [ ]:
def run_eval() -> None:
    passed = 0
    for case in CASES:
        output = str(agent(case.prompt))
        ok = case.check(output)
        status = "PASS" if ok else "FAIL"
        print(f"[{status}] {case.name}")
        if not ok:
            print(f"       prompt: {case.prompt}")
            print(f"       output: {output.strip()[:200]}")
        passed += ok

    print(f"\n{passed}/{len(CASES)} cases passed")


run_eval()
